In [ ]:
import pandas as pd
from pandas import DataFrame

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [ ]:
df_lessons_by_dates: DataFrame = pd.read_parquet("./df_lessons_by_dates.parquet")
df_lessons_by_dates.head()

In [ ]:
df_lessons_by_dates["visit_date"] = pd.to_datetime(df_lessons_by_dates["visit_date"])
df_lessons_by_dates.head()

In [ ]:
df_lessons_by_dates.info()

In [ ]:
def get_month_name(month_number: int) -> str:
    if isinstance(month_number, str):
        return month_number
    months: list[str] = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"]
    return months[month_number - 1]

def get_weekday_name(weekday_index: int) -> str:
    if isinstance(weekday_index, str):
        return weekday_index
    weekdays: list[str] = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    return weekdays[weekday_index]

def get_quarter_name(quarter_number: int) -> str:
    if isinstance(quarter_number, str):
        return quarter_number
    quarters: list[str] = ["Q1", "Q2", "Q3", "Q4"]
    return quarters[quarter_number - 1]

def get_hour_str(minutes_int: int) -> str:
    if isinstance(minutes_int, str):
        return minutes_int
    return str(minutes_int // 60) + ":00"

In [ ]:
df_lessons_by_dates["month"] = df_lessons_by_dates["month"].map(get_month_name)
df_lessons_by_dates["weekday"] = df_lessons_by_dates["weekday"].map(get_weekday_name)
df_lessons_by_dates["quarter"] = df_lessons_by_dates["quarter"].map(get_quarter_name)
#df_lessons_by_dates["hour"] = df_lessons_by_dates["minutes_begin"].map(get_hour_str)

In [ ]:
df_lessons_by_dates.head()

In [ ]:
df_lessons_by_dates.describe()

In [ ]:
df_lessons_by_dates.isna().sum()

In [ ]:
df_lessons_by_dates.duplicated().sum()

In [ ]:
bins = [0, 10, 20, 30, 40, 50, 60, 70]
labels = [
    "1-10",
    "11-20",
    "21-30",
    "31-40",
    "41-50",
    "51-60",
    "61-70"
]

df_distribution = (
    pd.cut(df_lessons_by_dates["visits_count"], bins=bins, labels=labels)
      .value_counts()
      .sort_index()
)

plt.figure(figsize=(8,4))

plt.bar(
    df_distribution.index.astype(str),
    df_distribution.values
)

plt.xlabel("Students per lesson")
plt.ylabel("Lessons")
plt.title("Distribution of visits_count")

plt.show()

### Forecast visits

In [ ]:
target: str = "visits_count"
df = df_lessons_by_dates.drop(
    columns=[
        "visit_date",
        "group_id",
        "style_id",
        "teacher_id",

        "minutes_begin", # no data for most part
        "minutes_begin", # to hours already
        "minutes_end",
        "amount"
    ]
)

In [ ]:
X = df.drop(columns=["visits_count"])
y = df["visits_count"]

In [ ]:
X.dtypes

In [ ]:
categorical_features = [
    "style_name",
    "teacher_full_name",
    "month",
    "weekday",
    "season",
    "quarter"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.6
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()]
)

plt.show()

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R²:", r2_score(y_test, y_pred))

- MAE = 2.77
В среднем модель ошибается менее чем на 3 человека. Если в среднем на занятии 8 человек, то ошибка в 2–3 человека вполне объяснима.

- RMSE = 3.79
Эта метрика сильнее штрафует большие ошибки. Иногда модель промахивается на 7–10 человек.

- R² = 0.53
Модель среднего уровня предсказания.

#### Try to improve model

In [ ]:
df2: DataFrame = df_lessons_by_dates
df2["day_of_month"] = "d" + df2["visit_date"].dt.day.astype(str)
df2["week_of_year"] = "w" + (df2["visit_date"].dt.isocalendar().week).astype(str)
df2

In [ ]:
df2 = df2.drop(
    columns=[
        "visit_date",
        "group_id",
        "style_id",
        "teacher_id",

        "minutes_begin", # no data for most part
        "minutes_begin", # to hours already
        "minutes_end",
        "amount"
    ]
)

X = df2.drop(columns=["visits_count"])
y = df2["visits_count"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X.dtypes

In [ ]:
preprocessor2 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            [
                "style_name",
                "teacher_full_name",
                "month",
                "weekday",
                "season",
                "quarter",

                "day_of_month",
                "week_of_year"
            ]
        )
    ],
    remainder="passthrough"
)

model2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor2),
        ("regressor", LinearRegression())
    ]
)
model2.fit(X_train, y_train)
y_pred = model2.predict(X_test)

In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.6
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()]
)

plt.show()

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R²:", r2_score(y_test, y_pred))

In [ ]:
feature_names = model2.named_steps["preprocessor"].get_feature_names_out()
coefficients = model2.named_steps["regressor"].coef_

coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coef": coefficients
    })
    .sort_values("coef", key=abs, ascending=False)
)

coef_df.head(20)

## Try trees

### Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
X = df.drop(columns=["visits_count"])
y = df["visits_count"]
X.dtypes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_tree = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            DecisionTreeRegressor(
                random_state=42,
                max_depth=6
            )
        )
    ]
)
model_tree.fit(X_train, y_train)
y_pred_tree = model_tree.predict(X_test)

In [ ]:
plt.figure(figsize=(8,8))

plt.scatter(
    y_test,
    y_pred_tree,
    alpha=0.6
)

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()]
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.title("Decision Tree")

plt.show()

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred_tree))
print("RMSE:", mean_squared_error(y_test, y_pred_tree) ** 0.5)
print("R²:", r2_score(y_test, y_pred_tree))

### Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                random_state=42,
                n_estimators=300,
                max_depth=10,
                n_jobs=-1
            )
        )
    ]
)

model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

In [ ]:
plt.figure(figsize=(8,8))

plt.scatter(
    y_test,
    y_pred_rf,
    alpha=0.6
)

plt.plot(
    [y.min(), y.max()],
    [y.min(), y.max()]
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.title("Random Forest")

plt.show()

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", mean_squared_error(y_test, y_pred_rf) ** 0.5)
print("R²:", r2_score(y_test, y_pred_rf))

In [ ]:
# Важность признаков
feature_names = model_rf.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = model_rf.named_steps[
    "regressor"
].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importance
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

importance_df.head(20)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_tree),
        mean_absolute_error(y_test, y_pred_rf)
    ],
    "RMSE": [
        mean_squared_error(y_test, y_pred) ** 0.5,
        mean_squared_error(y_test, y_pred_tree) ** 0.5,
        mean_squared_error(y_test, y_pred_rf) ** 0.5
    ],
    "R2": [
        r2_score(y_test, y_pred),
        r2_score(y_test, y_pred_tree),
        r2_score(y_test, y_pred_rf)
    ]
})

results.sort_values("R2", ascending=False)

# Conclusion

Attendance is determined primarily by:
1) teacher;
2) time of year;
3) style.

Three machine learning models were examined: linear regression, decision tree, and random forest. Linear regression showed the best forecasting quality (R² = 0.534, RMSE = 3.79, MAE = 2.77). More complex models did not improve the quality of the forecast, but on the contrary, showed worse results.

This indicates that the available features describe the relationship between class characteristics and attendance in a predominantly linear manner. Additionally, the relatively small sample size (1824 sessions) limits the power of more complex models.

Analysis of the importance of features showed that the teacher, seasonality and dance style have the greatest influence on the prognosis. Consequently, it is these factors that make the main contribution to the formation of class attendance.

# Forecast visits with schedule for now

In [ ]:
from src.load.internal.db.database_session import get_engine

engine = get_engine()

In [ ]:
df_schedules_data = pd.read_sql("""
SELECT schedules.id, schedules.day, schedules.group_id, 
styles.id AS style_id, styles.name AS style_name,
teachers.id AS teacher_id, CONCAT(teachers.last_name, ' ', teachers.name) AS teacher_full_name
	FROM public.schedules
INNER JOIN groups ON groups.id = schedules.group_id
INNER JOIN styles ON groups.style_id = styles.id
INNER JOIN teachers ON groups.teacher_id = teachers.id
WHERE groups.status = 1;
""", engine)
df_schedules_data

In [ ]:
future_dates = pd.DataFrame({
    "visit_date": pd.date_range(
        start="2026-08-01",
        end="2026-12-31",
        freq="D"
    )
})

future_dates["day"] = future_dates["visit_date"].dt.weekday + 1
future_dates

In [ ]:
future_lessons = future_dates.merge(
    df_schedules_data,
    on="day",
    how="inner"
)
future_lessons

In [ ]:
future_lessons["month"] = future_lessons["visit_date"].dt.month_name()
future_lessons["weekday"] = future_lessons["visit_date"].dt.day_name()
future_lessons["quarter"] = "Q" + future_lessons["visit_date"].dt.quarter.astype(str)

future_lessons["is_weekend"] = (
    future_lessons["visit_date"].dt.weekday >= 5
)

future_lessons["season"] = (
    future_lessons["month"]
        .map({
            "December": "winter",
            "January": "winter",
            "February": "winter",
            "March": "spring",
            "April": "spring",
            "May": "spring",
            "June": "summer",
            "July": "summer",
            "August": "summer",
            "September": "autumn",
            "October": "autumn",
            "November": "autumn"
        })
)

In [ ]:
X_future = future_lessons[
    [
        "style_name",
        "teacher_full_name",
        "month",
        "weekday",
        "is_weekend",
        "season",
        "quarter",
    ]
]
future_lessons["predicted_visits"] = np.maximum(
    0,
    model.predict(X_future).round()
)

In [ ]:
# forecast = (
#     future_lessons
#     .groupby("visit_date", as_index=False)["predicted_visits"]
#     .sum()
# )
# forecast
future_lessons.head(20)

In [ ]:
forecast = (
    future_lessons
    .groupby("visit_date", as_index=False)["predicted_visits"]
    .sum()
)
forecast.describe()

In [ ]:
history = (
    df_lessons_by_dates
    .groupby("visit_date", as_index=False)["visits_count"]
    .sum()
)
history.describe()

In [ ]:
df_lessons_by_dates.groupby("visit_date").size().describe()

In [ ]:
future_lessons.groupby("visit_date").size().describe()

In [ ]:
# pd.read_sql("""
# SELECT
#     status,
#     COUNT(*)
# FROM groups
# GROUP BY status;
# """, engine)

In [ ]:
def plot_teacher_styles(pairs: list[tuple[str, str]]):
    plt.figure(figsize=(16, 5))

    for teacher, style in pairs:
        data = future_lessons[
            (future_lessons["teacher_full_name"] == teacher) &
            (future_lessons["style_name"] == style)
        ].sort_values("visit_date")

        if data.empty:
            print(f"Нет данных: {style} — {teacher}")
            continue

        plt.plot(
            data["visit_date"],
            data["predicted_visits"],
            marker="o",
            label=f"{style} — {teacher}"
        )

    plt.title("Predicted visits by teacher and style")
    plt.xlabel("Date")
    plt.ylabel("Predicted visits")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
plot_teacher_styles([
    ("Teacher name", "Style name"),
])

In [ ]:
def plot_styles(styles: list[str]):
    plot_df = (
        future_lessons[
            future_lessons["style_name"].isin(styles)
        ]
        .groupby(["visit_date", "style_name"], as_index=False)
        ["predicted_visits"]
        .sum()
    )

    plt.figure(figsize=(16, 6))

    for style in styles:
        data = plot_df[
            plot_df["style_name"] == style
        ]

        plt.plot(
            data["visit_date"],
            data["predicted_visits"],
            label=style
        )

    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_styles([
    "HIGH-HEELS ",
    #"GIRLY HIP-HOP",
    "HIP-HOP "
])

In [ ]:
history = (
    df_lessons_by_dates[
        df_lessons_by_dates["style_name"] == "HIGH-HEELS "
    ]
    .groupby("visit_date", as_index=False)
    ["visits_count"]
    .sum()
)

forecast_style = (
    future_lessons[
        future_lessons["style_name"] == "HIGH-HEELS "
    ]
    .groupby("visit_date", as_index=False)
    ["predicted_visits"]
    .sum()
)

plt.figure(figsize=(16,6))

plt.plot(
    history["visit_date"],
    history["visits_count"],
    label="History"
)

plt.plot(
    forecast_style["visit_date"],
    forecast_style["predicted_visits"],
    label="Forecast"
)

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
style_forecast = (
    future_lessons
    .groupby(["visit_date", "style_name"], as_index=False)
    ["predicted_visits"]
    .sum()
)
styles = [
    "HIGH-HEELS",
    "GIRLY HIP-HOP",
    "CONTEMPORARY"
]
